# Fehlerbehandlung: Exceptions in Python

Bisher haben wir uns meist mit Code beschäftigt, der wie gewünscht funktioniert. In der Praxis läuft aber nicht immer alles nach Plan: Eine Datei fehlt, eine Nutzereingabe ist ungültig, eine Division durch Null wird versucht. Python signalisiert solche Probleme zur Laufzeit über sogenannte **Exceptions** (Ausnahmen) — vergleichbar mit einem Alarm, der auslöst, wenn während der Ausführung etwas nicht wie erwartet funktioniert. Ignorierst du den Alarm, stürzt dein Programm ab. Du kannst aber auch gezielt reagieren — das nennen wir **Fehlerbehandlung**.

Du kennst `try`/`except` bereits in Ansätzen aus den Sessions "Kontrollstrukturen" und "Dateien lesen und schreiben". In dieser Session schauen wir uns Exceptions im Detail an: was sie eigentlich sind, wie man sie gezielt abfängt, welche am häufigsten auftreten — und vor allem, wann man einen Fehler abfangen, wann man ihn bewusst crashen lassen und wann man selbst einen auslösen sollte.

## Agenda

In dieser Session schauen wir uns an:
1. **Was sind Exceptions?** — Fehlerarten, Tracebacks lesen
2. **Exceptions abfangen** — `try`/`except`, das Exception-Objekt, mehrere Exception-Typen
3. **Häufige Exception-Typen** — ein Überblick
4. **Eigene Exceptions auslösen** — `raise`
5. **Faustregeln** — Abfangen, crashen lassen oder selbst raisen?

## Inhaltsverzeichnis

- [1. Was sind Exceptions?](#1-was-sind-exceptions)
- [2. Exceptions abfangen mit `try`/`except`](#2-exceptions-abfangen-mit-tryexcept)
  - [2.1 Das Exception-Objekt und `as`](#21-das-exception-objekt-und-as)
  - [2.2 Mehrere Exception-Typen behandeln](#22-mehrere-exception-typen-behandeln)
- [3. Häufige Exception-Typen](#3-häufige-exception-typen)
- [4. Eigene Exceptions auslösen: `raise`](#4-eigene-exceptions-auslösen-raise)
  - [4.1 Erneut auslösen (Re-raise)](#41-erneut-auslösen-re-raise)
- [5. Faustregeln: Abfangen, crashen lassen oder selbst raisen?](#5-faustregeln-abfangen-crashen-lassen-oder-selbst-raisen)
- [6. Zusammenfassung](#6-zusammenfassung)
- [7. Vertiefung (Optional): `finally`, `else` und `assert`](#7-vertiefung-optional-finally-else-und-assert)

## 1. Was sind Exceptions?

Python unterscheidet grob zwei Arten von Fehlern:

- **Syntaxfehler** (`SyntaxError`): Der Code verstößt gegen die Grammatik der Sprache und kann gar nicht erst ausgeführt werden — vergleichbar mit einem Grammatikfehler in einem Satz. Zum Beispiel ein fehlender Doppelpunkt:
```python
if True
    print("Hallo")
```
- **Exceptions** (Ausnahmen): Der Code ist syntaktisch korrekt, aber **während der Ausführung** tritt ein Problem auf, das den normalen Ablauf unterbricht.

Schauen wir uns eine Exception in Aktion an:

In [ ]:
result = 10 / 0

Diese Fehlermeldung nennt man einen **Traceback**. Er verrät dir mehrere wichtige Dinge:

- **Wo** der Fehler aufgetreten ist (Datei und Zeile — bei mehreren verschachtelten Funktionsaufrufen sogar der komplette Aufrufpfad, von unten nach oben gelesen).
- **Welche Art** von Fehler aufgetreten ist (hier: `ZeroDivisionError`).
- **Welche Nachricht** dazu mitgegeben wurde (hier: "division by zero").

💡 Eine Exception ist, wie fast alles in Python, ein **Objekt** — mit einem Typ (z.B. `ZeroDivisionError`) und meist einer erklärenden Nachricht.

## 2. Exceptions abfangen mit `try`/`except`

Mit `try`/`except` kannst du auf eine Exception reagieren, statt das Programm abstürzen zu lassen. Der Code, der möglicherweise fehlschlägt, steht im `try`-Block. Tritt dort eine Exception des angegebenen Typs auf, wird die Ausführung des `try`-Blocks sofort abgebrochen und mit dem passenden `except`-Block fortgesetzt.

In [ ]:
try:
    result = 10 / 0
    print("Diese Zeile wird nicht mehr erreicht.")
except ZeroDivisionError:
    print("Division durch Null ist nicht erlaubt.")

print("Das Programm läuft normal weiter.")

⚠️ Gibst du hinter `except` **keinen** Exception-Typ an (`except:`), fängst du **jede** Exception ab — auch solche, die du eigentlich nicht erwartet oder gar nicht behandeln kannst. Das verschleiert Fehler, statt sie sichtbar zu machen. Gib deshalb (fast) immer einen konkreten Exception-Typ an.

### 2.1 Das Exception-Objekt und `as`

Mit `except ExceptionTyp as e` bekommst du das Exception-Objekt selbst in eine Variable (üblicherweise `e` genannt) und kannst z.B. seine Nachricht auslesen:

In [ ]:
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Ein Fehler ist aufgetreten: {e}")
    print(f"Exception-Typ: {type(e).__name__}")

### 2.2 Mehrere Exception-Typen behandeln

Ein `try`-Block kann mehrere unterschiedliche Exceptions auslösen, auf die du ggf. unterschiedlich reagieren willst. Dafür kannst du mehrere `except`-Blöcke hintereinander angeben:

In [ ]:
def safe_divide(dividend, divisor):
    try:
        return dividend / divisor
    except ZeroDivisionError:
        print("Division durch Null ist nicht erlaubt.")
    except TypeError:
        print("Beide Werte müssen Zahlen sein.")

safe_divide(10, 0)
safe_divide(10, "zwei")
print(safe_divide(10, 2))

Sollen mehrere Exception-Typen aber **gleich** behandelt werden, kannst du sie stattdessen als Tupel in einem einzigen `except` zusammenfassen:

In [ ]:
def safe_divide(dividend, divisor):
    try:
        return dividend / divisor
    except (ZeroDivisionError, TypeError) as e:
        print(f"Division nicht möglich: {e}")

safe_divide(10, 0)
safe_divide(10, "zwei")

## 3. Häufige Exception-Typen

Python bringt viele eingebaute Exception-Typen mit. Die häufigsten, die dir in der Praxis begegnen werden:

- **`ValueError`**: Ein Wert hat den richtigen Typ, aber einen ungültigen Inhalt (z.B. `int("abc")`).
- **`TypeError`**: Eine Operation wird auf einen Typ angewendet, der dafür nicht geeignet ist (z.B. `"5" + 5`).
- **`IndexError`**: Zugriff auf einen Listen-/Tupel-Index, der nicht existiert.
- **`KeyError`**: Zugriff auf einen Dictionary-Schlüssel, der nicht existiert.
- **`ZeroDivisionError`**: Division (oder Modulo) durch `0`.
- **`AttributeError`**: Zugriff auf ein Attribut oder eine Methode, die ein Objekt nicht besitzt.
- **`NameError`**: Verwendung eines Variablen- oder Funktionsnamens, der nicht definiert ist.
- **`FileNotFoundError`**: Kennst du bereits aus der Session "Dateien lesen und schreiben" — Zugriff auf eine Datei, die nicht existiert.

Schauen wir uns ein paar davon direkt an:

In [ ]:
int("abc")

In [ ]:
numbers = [1, 2, 3]
numbers[10]

In [ ]:
person = {"name": "Anna"}
person["age"]

💡 `TypeError`, `AttributeError` und `NameError` funktionieren nach demselben Prinzip — probier sie bei Bedarf selbst aus, z.B. mit `"5" + 5`, `"hallo".nicht_vorhanden()` oder dem Aufruf einer nie definierten Variable.

## 4. Eigene Exceptions auslösen: `raise`

Nicht nur Python selbst löst Exceptions aus — auch du kannst das gezielt tun, wenn deine eigene Funktion erkennt, dass etwas nicht stimmt. Das ist besonders nützlich, um **ungültige Eingaben** früh und mit einer klaren Fehlermeldung zurückzuweisen, statt später mit falschen Werten weiterzurechnen.

In [ ]:
def calculate_average(numbers):
    """Berechnet den Durchschnitt einer Liste von Zahlen."""
    if not numbers:
        raise ValueError("Die Liste darf nicht leer sein.")
    return sum(numbers) / len(numbers)

print(calculate_average([10, 20, 30]))

In [ ]:
calculate_average([])

Die aufrufende Stelle kann diese Exception nun ganz normal mit `try`/`except` behandeln:

In [ ]:
try:
    average = calculate_average([])
except ValueError as e:
    print(f"Ungültige Eingabe: {e}")
    average = 0

print(f"{average = }")

### 4.1 Erneut auslösen (Re-raise)

Manchmal willst du auf eine Exception reagieren (z.B. sie protokollieren), sie aber trotzdem weiterhin nach oben durchreichen, damit sie dort behandelt werden kann. Dafür genügt ein `raise` ohne Argument innerhalb des `except`-Blocks:

In [ ]:
def calculate_average_logged(numbers):
    try:
        return calculate_average(numbers)
    except ValueError:
        print("Achtung: calculate_average() wurde mit ungültigen Daten aufgerufen.")
        raise  # Löst dieselbe Exception erneut aus

calculate_average_logged([])

## 5. Faustregeln: Abfangen, crashen lassen oder selbst raisen?

Nicht jede Exception sollte abgefangen werden, und nicht jeder Fehlerfall sollte einfach durchlaufen. Ein paar Faustregeln, die dir bei der Entscheidung helfen:

- **Abfangen**, wenn du eine sinnvolle, konkrete Reaktion auf **genau diesen** Fehlerfall hast — z.B. einen Standardwert verwenden, es erneut versuchen oder eine verständliche Meldung statt eines Tracebacks ausgeben. Beispiel: `FileNotFoundError` beim Einlesen einer optionalen Konfigurationsdatei.
- **Crashen lassen** (also *nicht* abfangen), wenn die Exception auf einen **Programmierfehler** hinweist, den du eigentlich beheben solltest, statt ihn zu verschleiern. Ein `TypeError`, weil eine Funktion falsch aufgerufen wurde, soll dir laut auffallen — nicht in einem `except Exception: pass` verschwinden.
- **Selbst raisen**, wenn *deine* Funktion erkennt, dass eine Vorbedingung verletzt ist, und die aufrufende Stelle darüber informiert werden muss — z.B. bei einer ungültigen Eingabe wie in Kapitel 4.

⚠️ Das größte Antipattern der Fehlerbehandlung: Exceptions "verschlucken", indem man sie breit abfängt und nichts (oder nur `pass`) damit tut. So werden Fehler unsichtbar und tauchen später an ganz anderer Stelle als verwirrendes Folgeproblem wieder auf.

💡 Merkregel: Fange nur Exceptions ab, auf die du **an dieser Stelle im Code** sinnvoll reagieren kannst. Alles andere lässt du bewusst weiterlaufen (propagieren) — notfalls bis ganz nach oben, wo es zu einem sichtbaren Programmabsturz mit Traceback kommt. Das ist kein Bug, sondern gewolltes Verhalten: Ein sichtbarer Crash ist fast immer besser als ein Programm, das mit falschen Daten unbemerkt weiterläuft.

## 6. Zusammenfassung

In dieser Session haben wir uns mit Exceptions in Python beschäftigt:

- **Exceptions** sind Laufzeitfehler — Objekte mit einem Typ (z.B. `ValueError`) und einer Nachricht, die den normalen Programmablauf unterbrechen, wenn sie nicht abgefangen werden.
- **`try`/`except`** fängt eine Exception ab; mit `except Typ as e` bekommst du Zugriff auf das Exception-Objekt selbst.
- Häufige Exception-Typen: `ValueError`, `TypeError`, `IndexError`, `KeyError`, `ZeroDivisionError`, `AttributeError`, `NameError`, `FileNotFoundError`.
- **`raise`** löst selbst eine Exception aus, z.B. um ungültige Eingaben in eigenen Funktionen zurückzuweisen; ein bloßes `raise` im `except`-Block löst die aktuelle Exception erneut aus.
- Fange Exceptions gezielt ab, wenn du sinnvoll reagieren kannst — lass sie sonst bewusst crashen, statt sie zu verschlucken.

## 7. Vertiefung (Optional): `finally`, `else` und `assert`

📚 **Optional / Vertiefung** — für den Live-Vortrag bei Zeitdruck überspringbar, zum Nacharbeiten aber nützlich.

### `finally`

Ein `finally`-Block wird **immer** ausgeführt — egal, ob im `try`-Block eine Exception aufgetreten ist oder nicht. Das eignet sich für Aufräumarbeiten, die auf jeden Fall passieren müssen (z.B. eine Ressource freigeben):

In [ ]:
def process(value):
    try:
        print(f"Verarbeite {value}...")
        result = 100 / value
        print(f"Ergebnis: {result}")
    except ZeroDivisionError:
        print("Division durch Null übersprungen.")
    finally:
        print("Verarbeitung beendet.\n")

process(10)
process(0)

### `else` bei `try`

Ein `else`-Block nach `try`/`except` wird nur ausgeführt, wenn **keine** Exception aufgetreten ist — ähnlich wie du es schon von `while`-Schleifen kennst:

In [ ]:
try:
    result = 10 / 2
except ZeroDivisionError:
    print("Division durch Null ist nicht erlaubt.")
else:
    print(f"Berechnung erfolgreich: {result}")

### `assert`

Mit `assert` kannst du eine Bedingung prüfen, die eigentlich **immer** wahr sein sollte — ist sie es nicht, deutet das auf einen Bug in deinem eigenen Code hin, nicht auf eine ungültige Nutzereingabe. Ist die Bedingung `False`, löst `assert` eine `AssertionError`-Exception aus:

In [ ]:
def calculate_average(numbers):
    average = sum(numbers) / len(numbers)
    assert average >= 0, "Durchschnitt sollte hier nie negativ sein."
    return average

print(calculate_average([1, 2, 3]))

⚠️ Verwende `assert` nicht, um Nutzereingaben zu validieren — dafür ist `raise` (siehe Kapitel 4) das richtige Werkzeug. `assert`-Anweisungen können beim optimierten Ausführen von Python (`python -O`) sogar komplett deaktiviert werden und sollten deshalb nur interne Annahmen ("das darf laut meiner eigenen Logik nie passieren") prüfen, nicht Fehlerfälle, mit denen du rechnen musst.

💡 **Eigene Exception-Typen** (z.B. eine eigene `InsufficientFundsError`-Klasse) kannst du mit dem `class`-Schlüsselwort definieren — das lernst du in der Advanced Session "Fehlerbehandlung mit Exceptions".